[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/robot_arm/robot_arm.ipynb)

# A Robot Arm: Kinematics without a Jacobian

This example drives a three-joint arm: a base that turns about the vertical, and an elbow and a wrist that pitch. Where the tip goes for given joint angles is forward kinematics; which angles put the tip on a target is inverse kinematics; how fast the tip moves for given joint rates, and how hard each joint must hold against a force at the tip, are its velocity and statics. Here the arm homes onto a loop and then follows it round.

In projective geometric algebra a joint's axis is a line, a bivector, and its configuration is that line scaled by the angle; forward kinematics exponentiates those in order, each joint acting in the frame of the joints before it. Carried into those frames, the axes are the columns of the Jacobian, so it is never derived: a twist's commutator with the tip is the tip's velocity, and a force at the tip, a forque, loads each joint by its pairing with the carried axis, `axis & forque`. Inverse kinematics fits the tip's displacement to target as a combination of the columns' velocities, by least squares, and steps the joints along their axes.

In the matrix notation of robotics the carried axes read as the columns of the geometric Jacobian $J$, the tip velocity as $J\dot{\mathbf q}$, and the joint torques as $J^{\mathsf T}\mathbf F$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import concatenate, stack
from numga import NumpyContext
from numga.algebras import PGA3D as ga
from examples.util.animation import show_animation
from examples.mechanics.robot_arm import render, scenarios

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()
# A joint's axis scaled by an angle or a rate: a twist, the meet of two planes; and a force line, a forque.
Twist = ga.gatype.bivector()
Forque = ga.gatype.antibivector()
Motor = ga.gatype.rotor()
Scalar = ga.gatype.scalar()

## 1. The arm, and its kinematics

Each axis is the meet of two planes: the base's the vertical through the origin, the elbow's and wrist's horizontal lines at their heights. A joint's state is its axis times its angle. Each joint's step is the exponential of half its state, and the steps multiply on the right, each acting in the frame the ones before it leave: their running product, the reverse of the running product of their reverses.

In [ ]:
# The elbow's, wrist's and tip's heights in the home pose, and the rest angles of the base yaw and two pitches.
elbow_height, wrist_height, tip_height = 1.0, 2.0, 3.0
rest_angles = np.array([0.3, 0.5, 0.8])

In [ ]:
def forward_kinematics(joints: Twist) -> tuple[Motor, Motor]:
    """Return the tip pose and the frame before each joint's rotation."""
    steps = (joints * 0.5).exp()                                               # [joints] Motor
    # The steps multiply on the right: their running product is the reverse of the running product of
    # their reverses.
    poses = steps.reverse().cumprod(axis=0).reverse()                          # [joints] Motor
    return poses[-1], concatenate([mv.rotor().reshape(1), poses[:-1]])


def mechanics(joints: Twist, axis: Twist, rates: Twist, tip_home: Point,
              forque: Forque) -> tuple[Point, Scalar]:
    """Compute tip velocity and joint torques at the supplied configuration."""
    pose, frames = forward_kinematics(joints)
    tip = pose >> tip_home

    # Carry each joint's rate into its frame and sum to get the world twist.
    # Its commutator with the tip gives the tip velocity.
    twist = (frames >> rates).sum(axis=0)
    velocity = twist.commutator(tip)

    # Pair each carried axis with the applied forque: work per unit joint rate.
    joint_torques = (frames >> axis) & forque
    return velocity, joint_torques


axis = stack([mv.y ^ mv.x, mv.x ^ (mv.z - elbow_height * mv.w), mv.x ^ (mv.z - wrist_height * mv.w)])   # [joints] Twist
tip_home = mv.point([0.0, 0.0, tip_height])                                    # [] Point
rest = axis * rest_angles                                                      # [joints] Twist
pose, frames = forward_kinematics(rest)                                        # [] Motor, [joints] Motor

## 2. Velocity and statics

Each joint's rate, carried into its frame, adds to the arm's twist, and its commutator with the tip is the tip's velocity: the derivative of the tip's position along the rates, which a finite step confirms. A force at a point is a forque, the point joined with the weighted direction, and each joint's torque is the forque paired with the carried axis: the work it does per unit rate of that joint.

In [ ]:
# The joint rates, the tip force's point and direction, and the finite step the velocity is checked with.
joint_rates = np.array([0.5, -1.0, 0.25])
load_at, load = [1.0, 0.0, 3.0], [0.0, 2.0, -1.0]
difference = 1e-6

In [ ]:
rates = axis * joint_rates                                                     # [joints] Twist
forque = mv.point(load_at) & mv.direction(load)                                # [] Forque
velocity, joint_torques = mechanics(rest, axis, rates, tip_home, forque)       # [] Point, [joints] Scalar
ahead, _ = forward_kinematics(rest + rates * difference)
predicted = (pose >> tip_home) + velocity * difference                         # [] Point

In [ ]:
print("tip velocity:\n", velocity.coefficient_table())
print("joint torques:", joint_torques.to_array())
print("finite step against the velocity:", ((ahead >> tip_home) & predicted).norm().to_array())

## 3. Homing and tracking

Inverse kinematics: at each step the carried axes' velocities at the tip are the columns, and the joints move along their axes by the least-squares fit of the displacement to the target. Eight steps home the arm onto the start of a tilted loop; two per target then follow the loop round, the tip's trail in red.

In [ ]:
def inverse_kinematics(joints: Twist, axis: Twist, tip_home: Point, target: Point,
                       iterations: int):
    """Solve towards a target; return updated joints, link poses and the tip."""
    for _ in range(iterations):
        pose, frames = forward_kinematics(joints)
        tip = pose >> tip_home
        # Tip velocity per unit rate, per joint.
        columns = (frames >> axis).commutator(tip)
        # Fit the desired tip displacement as a weighted sum of these velocities;
        # apply those weights as joint increments along the home-pose axes.
        joints = joints + axis * columns.lstsq(target - tip)
    pose, frames = forward_kinematics(joints)
    # Frame after each joint.
    link_motors = concatenate([frames[1:], pose.reshape(1)])
    return joints, link_motors, pose >> tip_home


def track(joints: Twist, axis: Twist, tip_home: Point, targets: Point,
          iterations: int):
    """Follow targets, yielding joints, link poses, target and tip after each solve."""
    for target in targets:
        joints, link_motors, tip = inverse_kinematics(joints, axis, tip_home, target, iterations)
        yield joints, link_motors, target, tip


target = scenarios.loop_targets()[0]                                           # [] Point
homed, _, tip = inverse_kinematics(rest, axis, tip_home, target, scenarios.N_HOMING_STEPS)
show_animation(render.animate_tracking(scenarios.tracking()), "robot_arm", scenarios.FRAME_DURATION)

In [ ]:
print("homing error:", (target & tip).norm().to_array())
print("homed joint angles:", ((homed | axis) / (axis | axis)).to_array())

In [ ]:
# checks
# The tip velocity is the derivative of forward kinematics along the joint rates; homing reaches the target.
assert ((ahead >> tip_home) & predicted).norm().to_array().max() < 1e-9
assert (target & tip).norm().to_array().max() < 1e-8